# Know your customers

One of the most common applications of KMeans is to get to know your customers. Take a very simple dataset that is Mall Customers to try to discover customer segmentations.

0. Import usuals librairies

In [21]:
#!pip install -q xgboost
#!pip install -q s3fs
 #!pip install -U kaleido

# Load in our libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "colab" # pour que colab ne bloque pas l'export svg

import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression, Ridge

# metrics
from sklearn.metrics import r2_score, accuracy_score, silhouette_score

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier, VotingRegressor, StackingClassifier,  StackingRegressor

from sklearn.svm import SVR

from xgboost import XGBRegressor, XGBClassifier

### ML non supervisé
from sklearn.cluster import KMeans
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Import the ```Mall_Customers.csv``` dataset

In [22]:
df = pd.read_csv("/content/drive/MyDrive/JedhaBootcamp/Mall_Customers.csv")

df.head()

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [23]:
df.describe()


,CustomerID,Age,Annual Income (k$),Spending Score (1-100)
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,38.850000,60.560000,50.200000
std,57.879185,13.969007,26.264721,25.823522
min,1.000000,18.000000,15.000000,1.000000
25%,50.750000,28.750000,41.500000,34.750000
50%,100.500000,36.000000,61.500000,50.000000
75%,150.250000,49.000000,78.000000,73.000000
max,200.000000,70.000000,137.000000,99.000000


In [24]:
df.isna().sum().sum() * 100

np.int64(0)

2. Remove the "CustomerID" variable from your dataset.

In [25]:
df = df.drop("CustomerID", axis=1)
df.head()

,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,Male,19,15,39
1,Male,21,15,81
2,Female,20,16,6
3,Female,23,16,77
4,Female,31,17,40


3. Make all the preprocessings

L'algo Kmeans exige que que
- toutes les var soient numériques
- toutes les var soient standardisées (StandardScaler)
## / ! \ pas de variable cible ni de plit car KMeans est non supervisé i.e., qu'il apprend directement sur tout le dataset.

In [26]:
X = df.copy()
# Colonnes catégorielles et numériques
cat_features = ["Genre"]
num_features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
preprocessor = ColumnTransformer([
    ("encoder", OneHotEncoder(drop="first"), cat_features),
    # drop="first" supprime la première catégorie pour éviter la redondance et la multicolinéarité :
    # on encode une variable à n catégories avec seulement (n−1) colonnes
    ("scaler", StandardScaler(), num_features)
])
X_preprocessed = preprocessor.fit_transform(X)

print("Préprocessing terminé.")
print(X_preprocessed[:5])

Préprocessing terminé.
[[ 1.         -1.42456879 -1.73899919 -0.43480148]
 [ 1.         -1.28103541 -1.73899919  1.19570407]
 [ 0.         -1.3528021  -1.70082976 -1.71591298]
 [ 0.         -1.13750203 -1.70082976  1.04041783]
 [ 0.         -0.56336851 -1.66266033 -0.39597992]]


4. We are going to build our clusters, but to do so, we need to know the optimum number of clusters we need. First use the ```Elbow``` method to see if we can see how many we need to take as a value for ```k```.

In [27]:
# 1. Compute inertia for k from 1 to 10
inertia_values = []
K_range = range(1, 11)
# trop de clusters => sur segmentation, calcul lourd, lisibilité faible

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_preprocessed)
    inertia_values.append(kmeans.inertia_)
    # inertia_ => arg de Kmeans
    # Inertia = Σ (distance(point_i, centroïde_cluster_i))²


# 2. Display inertia values
print(inertia_values)

[649.2800000000001, 438.5224115567773, 345.2050346991058, 254.36171423484495, 216.78490151651047, 181.9514362434146, 171.37266048943803, 153.29724063982405, 142.71846488584754, 133.3266400544609]


In [28]:
df_elbow = pd.DataFrame({
    "k": list(K_range),
    "inertia": inertia_values
})
df_elbow.head()

,k,inertia
0,1,649.280000
1,2,438.522412
2,3,345.205035
3,4,254.361714
4,5,216.784902


In [29]:


fig = px.line(
    df_elbow, # pazs
    x="k",
    y="inertia",
    markers=True,
    title="Elbow Method (Plotly)"
    # méthode graphique pour trouver le k optimal en observant où al réduction
    # d'inertie commence à diminuer fortement
)

fig.update_layout(
    xaxis_title="Number of clusters k",
    yaxis_title="Inertia (WCSS)",
    # With In Cluster of Sum Squares :
    # on calcule l'inertie pour chaque k
    # on trace la courbe k vs inertie
    template="plotly_white"
)

fig.show()

chercher le “coude” du graphique : point où

- la réduction d’inertie devient marginale

- la segmentation est déjà suffisamment fine

- augmenter k n'apporte presque plus d’information utile

Donc k = 6 est un très bon candidat comme nombre optimal de clusters.

5. Then use the _Silhouette_ method to see if we can refine our hypothesis for ```k```.

On ne calcule jamais *silhouette* pour k=1 car un cluster unique n’a pas de silhouette (distance intra/inter impossible à comparer)

La Silhouette Method mesure la qualité de séparation entre les clusters.

Elle ne doit jamais être utilisée seule → elle complète l’Elbow Method

Lorsque les deux méthodes convergent comme c'est le cas ici pour k=6, la décision est fiable.

In [30]:
silhouette_values = []

K_range = range(2, 11)  # On commence à 2

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_preprocessed)
    # fit_predict pour prédire les clusters, pas de labels pas de silhouette
    # elbow methods (fit simple) ne nécessitait pas de labels parce que l'inertie était basée sur les centres
    score = silhouette_score(X_preprocessed, labels)
    silhouette_values.append(score)

print(silhouette_values)

[np.float64(0.30319765641607577), np.float64(0.31200836318195724), np.float64(0.35044061449694197), np.float64(0.34977050035201074), np.float64(0.356485834425401), np.float64(0.3315513749667554), np.float64(0.336203797093241), np.float64(0.3117853954011984), np.float64(0.30866345259814115)]


In [31]:
df_sil = pd.DataFrame({
    "k": list(K_range),
    "silhouette": silhouette_values
})

fig = px.line(
    df_sil,
    x="k",
    y="silhouette",
    markers=True,
    title="Silhouette Method"
)

fig.update_layout(
    xaxis_title="Number of clusters k",
    yaxis_title="Silhouette Score",
    template="plotly_white"
)

fig.show()

6. Next, we will take $K=6$ clusters. Apply the KMeans to your dataset.

In [32]:
kmeans = KMeans(n_clusters=6, random_state=42)
labels = kmeans.fit_predict(X_preprocessed)
labels[:10]

array([4, 4, 2, 4, 2, 4, 2, 4, 2, 4], dtype=int32)

In [33]:
df['Cluster_KMeans'] = labels
df["Clusters_1_6"] = df["Cluster_KMeans"] + 1 # pour éviter que kmeans assigne par défaut des labels 0 aux clusters

df.head()


,Genre,Age,Annual Income (k$),Spending Score (1-100),Cluster_KMeans,Clusters_1_6
0,Male,19,15,39,4,5
1,Male,21,15,81,4,5
2,Female,20,16,6,2,3
3,Female,23,16,77,4,5
4,Female,31,17,40,2,3


- Si résultats à priori différents de ceux de Jedha  => soit random state # ou preprocessing #
- Si labels différents => ce n'est pas une différence de segmentation mais d'etiquettage non significative. Ce qui importe ici ce sont les centroides des clusters.

7. Let's create a graph that will allow us to visualize each of the clusters. We will first take the ```Spending Score``` as the ordinate and the ```Annual Income``` as the abscissa.

In [34]:
fig = px.scatter(
    df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    color="Cluster_KMeans",
    title="Customer Segmentation (Income vs Spending Score)",
    color_continuous_scale="Turbo",
    hover_data=["Genre", "Age", "Annual Income (k$)", "Spending Score (1-100)"]
)

fig.show()

KMeans a été entraîné sur 4 dimensions :

Genre (encodé)

Age (standardisé)

Income (standardisé)

Spending (standardisé)

or le graphique affiche une représentaion 2D d'un Kmeans 4D => visuellement on croit que 2 clusters se superposent alors que les ages des individus de ces 2 clusters sont totalements opposés

8. We have a nice visualization with a nice cluster separation. Look this time at the variable ```Age``` in relation to the ```Spending Score```. What do you notice?

In [35]:
fig = px.scatter(
    df,
    x="Age",
    y="Spending Score (1-100)",
    color="Clusters_1_6",
    title="Customer Segmentation (Age vs Spending Score)",
    color_continuous_scale="Turbo",
    hover_data=["Genre", "Annual Income (k$)", "Spending Score (1-100)", "Cluster_KMeans"]
)

fig.update_layout(
    xaxis_title="Age",
    yaxis_title="Spending Score (1-100)",
    template="plotly_white"
)

fig.show()

----> This time clusters are definitely less visible.
on ne peut pas se baser sur l'age pour la segmentation
Le spendinng_score ne dépend clairemnt pas de l'age

9. Finally, make a 3d scatter plot of your clusters by using all the quantitative features

In [36]:
fig = px.scatter_3d(
    df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    z="Age",
    color="Clusters_1_6",
    title="3D Visualization of Customer Segments (KMeans)",
    hover_data=["Genre", "Age", "Annual Income (k$)", "Spending Score (1-100)"],
    color_continuous_scale="Turbo"
)

fig.update_layout(
    scene=dict(
        xaxis_title="Annual Income (k$)",
        yaxis_title="Spending Score (1-100)",
        zaxis_title="Age"
    ),
    template="plotly_white"
)

fig.show()

la projection 3D en révèle beaucoup plus que les projections 2D.